In [7]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import gc

# Chemin vers le fichier brut
RAW_PATH = Path("../data/raw/pfas_data_hub")

# Dossier de sortie pour les données staging
STAGING_DIR = Path("../data/staging")

# Création du dossier staging s'il n'existe pas
STAGING_DIR.mkdir(parents=True, exist_ok=True)

print("Chemin RAW :", RAW_PATH)
print("Dossier STAGING :", STAGING_DIR)

Chemin RAW : ..\data\raw\pfas_data_hub
Dossier STAGING : ..\data\staging


In [8]:
# Chargement du dataset brut depuis la zone RAW
df_raw = pd.read_parquet(RAW_PATH)

# Affichage du volume des données
print("Données brutes chargées")
print("Nombre de lignes :", df_raw.shape[0])
print("Nombre de colonnes :", df_raw.shape[1])

# Aperçu des premières lignes
display(df_raw.head())

Données brutes chargées
Nombre de lignes : 942084
Nombre de colonnes : 21


,category,lat,lon,name,city,country,type,sector,source_type,data_collection_method,...,source_url,dataset_id,dataset_name,pfas_values,unit,pfas_sum,details,matrix,date,year
0,Known PFAS user,45.864286,5.948318,Fortier Beaulieu,Rumilly,France,Industrial site,NaN,Authorities,OSINT,...,https://www.auvergne-rhone-alpes.ars.sante.fr/...,0,Known PFAS Users,[],NaN,NaN,{},NaN,NaN,NaN
1,Known PFAS user,45.838896,5.960591,Salomon,Rumilly,France,Industrial site,NaN,Authorities,OSINT,...,https://www.auvergne-rhone-alpes.ars.sante.fr/...,0,Known PFAS Users,[],NaN,NaN,{},NaN,NaN,NaN
2,Known PFAS user,46.566759,4.904980,Tefal,Tournus,France,Industrial site,NaN,Authorities,OSINT,...,https://www.georisques.gouv.fr/risques/registr...,0,Known PFAS Users,[],NaN,NaN,{},NaN,NaN,NaN
3,Known PFAS user,47.984030,14.272978,Agru,Waldneukirchen,Austria,Industrial site,Manufacture of rubber and plastic products,Company,OSINT,...,https://www.agru.at/en/,0,Known PFAS Users,[],NaN,NaN,"{""maps_link"": ""https://goo.gl/maps/cSD9QcvjzUh...",NaN,NaN,NaN
4,Known PFAS user,48.032000,14.225225,Agru,Bad Hall,Austria,Industrial site,Manufacture of rubber and plastic products,Company,OSINT,...,https://www.agru.at/en/,0,Known PFAS Users,[],NaN,NaN,"{""maps_link"": ""https://goo.gl/maps/NkuHEHiEQbV...",NaN,NaN,NaN


In [9]:
def empty_list_to_nan(x):
    # Transforme les listes vides [] en valeur manquante
    if pd.isna(x):
        return np.nan
    
    if isinstance(x, list) and len(x) == 0:
        return np.nan
    
    if isinstance(x, str) and x.strip() == "[]":
        return np.nan
    
    return x


def parse_pfas_values(x):
    # Transforme le texte JSON de pfas_values en liste Python
    if pd.isna(x):
        return []
    
    if isinstance(x, list):
        return x
    
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return []
    
    return []


def has_valid_coordinates(lat, lon):
    # Vérifie si les coordonnées géographiques sont valides
    if pd.isna(lat) or pd.isna(lon):
        return False
    
    return (-90 <= lat <= 90) and (-180 <= lon <= 180)

In [10]:
# On garde uniquement les lignes qui correspondent à des mesures
df_measurements = df_raw[df_raw["category"] == "Measurement"].copy()

# Réinitialisation de l'index
df_measurements = df_measurements.reset_index(drop=True)

# Création d'un identifiant unique pour chaque observation
df_measurements.insert(0, "observation_id", df_measurements.index + 1)

print("Nombre d'observations Measurement :", df_measurements.shape[0])

display(df_measurements.head())

Nombre d'observations Measurement : 929442


,observation_id,category,lat,lon,name,city,country,type,sector,source_type,...,source_url,dataset_id,dataset_name,pfas_values,unit,pfas_sum,details,matrix,date,year
0,1,Measurement,50.808932,3.352552,Maes,Zwevegem,Belgium,Industrial site,Finishing of textiles,Authorities,...,https://drive.google.com/drive/folders/1CKxJ5Q...,10,15 textile facilities emitting PFAS,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...",ng/l,130.0,{},Surface water,NaN,2018.0
1,2,Measurement,51.016507,4.088303,Tarkett,Dendermonde,Belgium,Industrial site,Finishing of textiles,Authorities,...,https://drive.google.com/drive/folders/1CKxJ5Q...,10,15 textile facilities emitting PFAS,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...",ng/l,200.0,{},Surface water,NaN,2017.0
2,3,Measurement,51.042282,3.548967,Ververijen Escotex,Deinze,Belgium,Industrial site,Finishing of textiles,Authorities,...,https://drive.google.com/drive/folders/1CKxJ5Q...,10,15 textile facilities emitting PFAS,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...",ng/l,42400.0,{},Surface water,NaN,2016.0
3,4,Measurement,51.771554,6.605953,Gerhard Van Clewe,Hamminkeln-Dingden,Germany,Industrial site,Finishing of textiles,Authorities,...,https://drive.google.com/drive/folders/1CKxJ5Q...,10,15 textile facilities emitting PFAS,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...",ng/l,50.0,{},Surface water,NaN,2017.0
4,5,Measurement,49.590101,7.603395,KOB Karl Otto Braun,Wolfstein,Germany,Industrial site,Finishing of textiles,Authorities,...,https://drive.google.com/drive/folders/1CKxJ5Q...,10,15 textile facilities emitting PFAS,"[{""cas_id"": ""307-24-4"", ""unit"": ""ng/l"", ""subst...",ng/l,580.0,{},Surface water,NaN,2018.0


In [11]:
# Colonnes utiles pour la table observation
observation_cols = [
    "observation_id",
    "dataset_id",
    "dataset_name",
    "name",
    "country",
    "city",
    "lat",
    "lon",
    "year",
    "date",
    "category",
    "matrix",
    "unit",
    "pfas_sum",
    "pfas_values"
]

# On garde seulement les colonnes qui existent dans le dataset
observation_cols = [col for col in observation_cols if col in df_measurements.columns]

# Création de la table observations
df_observations = df_measurements[observation_cols].copy()

print("Taille de la table observations :", df_observations.shape)

display(df_observations.head())

Taille de la table observations : (929442, 15)


,observation_id,dataset_id,dataset_name,name,country,city,lat,lon,year,date,category,matrix,unit,pfas_sum,pfas_values
0,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,Measurement,Surface water,ng/l,130.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst..."
1,2,10,15 textile facilities emitting PFAS,Tarkett,Belgium,Dendermonde,51.016507,4.088303,2017.0,NaN,Measurement,Surface water,ng/l,200.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst..."
2,3,10,15 textile facilities emitting PFAS,Ververijen Escotex,Belgium,Deinze,51.042282,3.548967,2016.0,NaN,Measurement,Surface water,ng/l,42400.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst..."
3,4,10,15 textile facilities emitting PFAS,Gerhard Van Clewe,Germany,Hamminkeln-Dingden,51.771554,6.605953,2017.0,NaN,Measurement,Surface water,ng/l,50.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst..."
4,5,10,15 textile facilities emitting PFAS,KOB Karl Otto Braun,Germany,Wolfstein,49.590101,7.603395,2018.0,NaN,Measurement,Surface water,ng/l,580.0,"[{""cas_id"": ""307-24-4"", ""unit"": ""ng/l"", ""subst..."


In [12]:
# Nettoyage de pfas_values : les listes vides deviennent NaN
df_observations["pfas_values_clean"] = df_observations["pfas_values"].apply(empty_list_to_nan)

# Indique si une observation contient des détails PFAS exploitables
df_observations["has_pfas_details"] = df_observations["pfas_values_clean"].notna()

# Conversion de pfas_sum en valeur numérique
df_observations["pfas_sum"] = pd.to_numeric(
    df_observations["pfas_sum"],
    errors="coerce"
)

# Indicateur : somme PFAS positive
df_observations["is_positive_pfas_sum"] = df_observations["pfas_sum"] > 0

# Indicateur : somme PFAS égale à zéro
df_observations["is_zero_pfas_sum"] = df_observations["pfas_sum"] == 0

# Vérification des coordonnées géographiques
df_observations["has_valid_coordinates"] = df_observations.apply(
    lambda row: has_valid_coordinates(row.get("lat"), row.get("lon")),
    axis=1
)

print("Taux d'observations avec détails PFAS :")
print(round(df_observations["has_pfas_details"].mean() * 100, 2), "%")

print("Taux de valeurs pfas_sum positives :")
print(round(df_observations["is_positive_pfas_sum"].mean() * 100, 2), "%")

display(df_observations.head())

Taux d'observations avec détails PFAS :
100.0 %
Taux de valeurs pfas_sum positives :
38.87 %


,observation_id,dataset_id,dataset_name,name,country,city,lat,lon,year,date,category,matrix,unit,pfas_sum,pfas_values,pfas_values_clean,has_pfas_details,is_positive_pfas_sum,is_zero_pfas_sum,has_valid_coordinates
0,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,Measurement,Surface water,ng/l,130.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...","[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...",True,True,False,True
1,2,10,15 textile facilities emitting PFAS,Tarkett,Belgium,Dendermonde,51.016507,4.088303,2017.0,NaN,Measurement,Surface water,ng/l,200.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...","[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...",True,True,False,True
2,3,10,15 textile facilities emitting PFAS,Ververijen Escotex,Belgium,Deinze,51.042282,3.548967,2016.0,NaN,Measurement,Surface water,ng/l,42400.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...","[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...",True,True,False,True
3,4,10,15 textile facilities emitting PFAS,Gerhard Van Clewe,Germany,Hamminkeln-Dingden,51.771554,6.605953,2017.0,NaN,Measurement,Surface water,ng/l,50.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...","[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst...",True,True,False,True
4,5,10,15 textile facilities emitting PFAS,KOB Karl Otto Braun,Germany,Wolfstein,49.590101,7.603395,2018.0,NaN,Measurement,Surface water,ng/l,580.0,"[{""cas_id"": ""307-24-4"", ""unit"": ""ng/l"", ""subst...","[{""cas_id"": ""307-24-4"", ""unit"": ""ng/l"", ""subst...",True,True,False,True


In [13]:
# On retire les colonnes lourdes inutiles dans la table staging finale
df_observations_staging = df_observations.drop(
    columns=["pfas_values", "pfas_values_clean"],
    errors="ignore"
)

print("Taille finale table observations STAGING :", df_observations_staging.shape)

display(df_observations_staging.head())

Taille finale table observations STAGING : (929442, 18)


,observation_id,dataset_id,dataset_name,name,country,city,lat,lon,year,date,category,matrix,unit,pfas_sum,has_pfas_details,is_positive_pfas_sum,is_zero_pfas_sum,has_valid_coordinates
0,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,Measurement,Surface water,ng/l,130.0,True,True,False,True
1,2,10,15 textile facilities emitting PFAS,Tarkett,Belgium,Dendermonde,51.016507,4.088303,2017.0,NaN,Measurement,Surface water,ng/l,200.0,True,True,False,True
2,3,10,15 textile facilities emitting PFAS,Ververijen Escotex,Belgium,Deinze,51.042282,3.548967,2016.0,NaN,Measurement,Surface water,ng/l,42400.0,True,True,False,True
3,4,10,15 textile facilities emitting PFAS,Gerhard Van Clewe,Germany,Hamminkeln-Dingden,51.771554,6.605953,2017.0,NaN,Measurement,Surface water,ng/l,50.0,True,True,False,True
4,5,10,15 textile facilities emitting PFAS,KOB Karl Otto Braun,Germany,Wolfstein,49.590101,7.603395,2018.0,NaN,Measurement,Surface water,ng/l,580.0,True,True,False,True


In [14]:
# Colonnes utiles pour construire la table détaillée par substance
detail_cols = [
    "observation_id",
    "dataset_id",
    "dataset_name",
    "name",
    "country",
    "city",
    "lat",
    "lon",
    "year",
    "date",
    "category",
    "matrix",
    "unit",
    "pfas_sum",
    "pfas_values_clean"
]

# On garde seulement les colonnes disponibles
detail_cols = [col for col in detail_cols if col in df_observations.columns]

# On garde uniquement les observations qui ont des détails PFAS
df_work = df_observations[
    df_observations["has_pfas_details"]
][detail_cols].copy()

print("Nombre d'observations avec détails PFAS :", df_work.shape[0])

display(df_work.head())

Nombre d'observations avec détails PFAS : 929442


,observation_id,dataset_id,dataset_name,name,country,city,lat,lon,year,date,category,matrix,unit,pfas_sum,pfas_values_clean
0,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,Measurement,Surface water,ng/l,130.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst..."
1,2,10,15 textile facilities emitting PFAS,Tarkett,Belgium,Dendermonde,51.016507,4.088303,2017.0,NaN,Measurement,Surface water,ng/l,200.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst..."
2,3,10,15 textile facilities emitting PFAS,Ververijen Escotex,Belgium,Deinze,51.042282,3.548967,2016.0,NaN,Measurement,Surface water,ng/l,42400.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst..."
3,4,10,15 textile facilities emitting PFAS,Gerhard Van Clewe,Germany,Hamminkeln-Dingden,51.771554,6.605953,2017.0,NaN,Measurement,Surface water,ng/l,50.0,"[{""cas_id"": ""335-67-1"", ""unit"": ""ng/l"", ""subst..."
4,5,10,15 textile facilities emitting PFAS,KOB Karl Otto Braun,Germany,Wolfstein,49.590101,7.603395,2018.0,NaN,Measurement,Surface water,ng/l,580.0,"[{""cas_id"": ""307-24-4"", ""unit"": ""ng/l"", ""subst..."


In [15]:
# Transformation de pfas_values_clean en vraie liste Python
df_work["pfas_values_parsed"] = df_work["pfas_values_clean"].apply(parse_pfas_values)

# Nombre de substances PFAS par observation
display(df_work["pfas_values_parsed"].apply(len).describe())

count    929442.000000
mean         23.165422
std          16.654867
min           1.000000
25%           8.000000
50%          22.000000
75%          35.000000
max          96.000000
Name: pfas_values_parsed, dtype: float64

In [16]:
# Suppression de la colonne texte avant explode pour économiser la mémoire
df_work = df_work.drop(columns=["pfas_values_clean"])

print("Taille de la table de travail avant explode :", df_work.shape)

Taille de la table de travail avant explode : (929442, 15)


In [17]:
# Chaque substance PFAS devient une ligne séparée
df_long = df_work.explode("pfas_values_parsed", ignore_index=True)

print("Taille après explode :", df_long.shape)

display(df_long.head())

Taille après explode : (21530916, 15)


,observation_id,dataset_id,dataset_name,name,country,city,lat,lon,year,date,category,matrix,unit,pfas_sum,pfas_values_parsed
0,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,Measurement,Surface water,ng/l,130.0,"{'cas_id': '335-67-1', 'unit': 'ng/l', 'substa..."
1,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,Measurement,Surface water,ng/l,130.0,"{'cas_id': '1763-23-1', 'unit': 'ng/l', 'subst..."
2,2,10,15 textile facilities emitting PFAS,Tarkett,Belgium,Dendermonde,51.016507,4.088303,2017.0,NaN,Measurement,Surface water,ng/l,200.0,"{'cas_id': '335-67-1', 'unit': 'ng/l', 'substa..."
3,3,10,15 textile facilities emitting PFAS,Ververijen Escotex,Belgium,Deinze,51.042282,3.548967,2016.0,NaN,Measurement,Surface water,ng/l,42400.0,"{'cas_id': '335-67-1', 'unit': 'ng/l', 'substa..."
4,3,10,15 textile facilities emitting PFAS,Ververijen Escotex,Belgium,Deinze,51.042282,3.548967,2016.0,NaN,Measurement,Surface water,ng/l,42400.0,"{'cas_id': '1763-23-1', 'unit': 'ng/l', 'subst..."


In [18]:
# Transformation des dictionnaires PFAS en colonnes
pfas_details = pd.json_normalize(df_long["pfas_values_parsed"])

# Suppression de la colonne JSON après normalisation
df_long = df_long.drop(columns=["pfas_values_parsed"])

# Fusion entre les informations générales et les détails PFAS
df_pfas_long_staging = pd.concat(
    [
        df_long.reset_index(drop=True),
        pfas_details.add_prefix("pfas_").reset_index(drop=True)
    ],
    axis=1
)

print("Taille finale table mesures longues STAGING :", df_pfas_long_staging.shape)

display(df_pfas_long_staging.head())

Taille finale table mesures longues STAGING : (21530916, 20)


,observation_id,dataset_id,dataset_name,name,country,city,lat,lon,year,date,category,matrix,unit,pfas_sum,pfas_cas_id,pfas_unit,pfas_substance,pfas_value,pfas_isomer,pfas_less_than
0,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,Measurement,Surface water,ng/l,130.0,335-67-1,ng/l,PFOA,90.0,NaN,NaN
1,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,Measurement,Surface water,ng/l,130.0,1763-23-1,ng/l,PFOS,40.0,NaN,NaN
2,2,10,15 textile facilities emitting PFAS,Tarkett,Belgium,Dendermonde,51.016507,4.088303,2017.0,NaN,Measurement,Surface water,ng/l,200.0,335-67-1,ng/l,PFOA,200.0,NaN,NaN
3,3,10,15 textile facilities emitting PFAS,Ververijen Escotex,Belgium,Deinze,51.042282,3.548967,2016.0,NaN,Measurement,Surface water,ng/l,42400.0,335-67-1,ng/l,PFOA,41400.0,NaN,NaN
4,3,10,15 textile facilities emitting PFAS,Ververijen Escotex,Belgium,Deinze,51.042282,3.548967,2016.0,NaN,Measurement,Surface water,ng/l,42400.0,1763-23-1,ng/l,PFOS,500.0,NaN,NaN


In [19]:
# Conversion de la valeur PFAS en format numérique
if "pfas_value" in df_pfas_long_staging.columns:
    df_pfas_long_staging["pfas_value"] = pd.to_numeric(
        df_pfas_long_staging["pfas_value"],
        errors="coerce"
    )

print("Statistiques de pfas_value :")

display(df_pfas_long_staging["pfas_value"].describe())

Statistiques de pfas_value :


count    1.165797e+06
mean     1.362051e+04
std      9.055518e+05
min      2.000000e-04
25%      3.000000e+00
50%      1.000000e+01
75%      5.363000e+01
max      5.090000e+08
Name: pfas_value, dtype: float64

In [20]:
# Résumé du volume des tables staging
quality_summary = pd.DataFrame({
    "table": ["observations_staging", "measurements_long_staging"],
    "rows": [df_observations_staging.shape[0], df_pfas_long_staging.shape[0]],
    "columns": [df_observations_staging.shape[1], df_pfas_long_staging.shape[1]]
})

display(quality_summary)

,table,rows,columns
0,observations_staging,929442,18
1,measurements_long_staging,21530916,20


In [21]:
# Calcul des valeurs manquantes dans la table détaillée
missing_measurements = pd.DataFrame({
    "column": df_pfas_long_staging.columns,
    "missing_values": df_pfas_long_staging.isna().sum().values,
    "missing_rate_%": (df_pfas_long_staging.isna().mean() * 100).round(2).values
}).sort_values("missing_rate_%", ascending=False)

display(missing_measurements)

,column,missing_values,missing_rate_%
18,pfas_isomer,21375630,99.28
17,pfas_value,20365119,94.59
5,city,12213523,56.73
19,pfas_less_than,1165797,5.41
7,lon,687861,3.19
6,lat,687861,3.19
9,date,132967,0.62
4,country,16413,0.08
3,name,0,0.00
2,dataset_name,0,0.00


In [22]:
# Substances PFAS les plus fréquentes
print("Top 20 substances PFAS :")
display(df_pfas_long_staging["pfas_substance"].value_counts().head(20))

# Répartition des unités dans les valeurs détaillées
print("Répartition des unités PFAS :")
display(df_pfas_long_staging["pfas_unit"].value_counts(dropna=False))

# Répartition des matrices
print("Répartition des matrices :")
display(df_pfas_long_staging["matrix"].value_counts(dropna=False))

Top 20 substances PFAS :


pfas_substance
Diflufenican          587929
Norflurazon           551757
Tetraconazole         536259
Fludioxonil           529705
Flurochloridone       517012
Flufenacet            506839
Flurtamone            502598
Flazasulfuron         488445
Isoxaflutole          485292
Prosulfuron           473530
Trifloxystrobin       467636
lambda-Cyhalothrin    466974
Benfluralin           463769
Fipronil              450256
Oxyfluorfen           436695
Picoxystrobin         428744
Tefluthrin            426335
Bifenthrin            425597
PFOS                  384857
Tau-Fluvalinate       383304
Name: count, dtype: int64

Répartition des unités PFAS :


pfas_unit
ng/l        20451992
ng/kg dw      840698
ng/kg         174668
ng/kg fw       63558
Name: count, dtype: int64

Répartition des matrices :


matrix
Surface water       11168449
Groundwater          8937134
Soil                  822943
Drinking water        207472
Sediment              175129
Wastewater            106026
Biota                  69543
Unknown                21134
Suspended matter       10862
Sea water               8361
Rainwater               2612
Leachate                1210
Sludge                    38
Sewerage                   3
Name: count, dtype: int64

In [23]:
# Fonction pour standardiser les unités
def standardize_unit(x):
    if pd.isna(x):
        return np.nan
    
    s = str(x).strip().lower()
    s = s.replace(" ", "")
    
    if s in ["ng/l", "ng/litre", "ng/liter"]:
        return "ng/L"
    
    if s in ["ng/kg"]:
        return "ng/kg"
    
    if s in ["ng/kgdw", "ng/kgdryweight"]:
        return "ng/kg dw"
    
    if s in ["ng/kgfw", "ng/kgfreshweight"]:
        return "ng/kg fw"
    
    return str(x).strip()


# Standardisation de l'unité globale dans la table observation
df_observations_staging["unit_standardized"] = df_observations_staging["unit"].apply(standardize_unit)

# Standardisation de l'unité globale dans la table longue
df_pfas_long_staging["unit_standardized"] = df_pfas_long_staging["unit"].apply(standardize_unit)

# Standardisation de l'unité détaillée PFAS
if "pfas_unit" in df_pfas_long_staging.columns:
    df_pfas_long_staging["pfas_unit_standardized"] = df_pfas_long_staging["pfas_unit"].apply(standardize_unit)

display(df_observations_staging["unit_standardized"].value_counts(dropna=False))

unit_standardized
ng/L        872923
ng/kg dw     40083
ng/kg        11579
ng/kg fw      4857
Name: count, dtype: int64

In [24]:
# Fonction pour standardiser les matrices environnementales
def standardize_matrix(x):
    if pd.isna(x):
        return "Unknown"
    
    s = str(x).strip().lower()
    
    matrix_map = {
        "groundwater": "Groundwater",
        "surface water": "Surface water",
        "wastewater": "Wastewater",
        "drinking water": "Drinking water",
        "sea water": "Sea water",
        "soil": "Soil",
        "sediment": "Sediment",
        "biota": "Biota",
        "leachate": "Leachate",
        "unknown": "Unknown"
    }
    
    return matrix_map.get(s, str(x).strip())


# Création d'une colonne standardisée pour la matrice
df_observations_staging["matrix_standardized"] = df_observations_staging["matrix"].apply(standardize_matrix)

df_pfas_long_staging["matrix_standardized"] = df_pfas_long_staging["matrix"].apply(standardize_matrix)

display(df_observations_staging["matrix_standardized"].value_counts(dropna=False))

matrix_standardized
Surface water       449509
Groundwater         394360
Soil                 33796
Sediment             15373
Wastewater           13287
Drinking water       12322
Biota                 5684
Unknown               1886
Suspended matter      1539
Sea water             1334
Rainwater              189
Leachate               149
Sludge                  11
Sewerage                 3
Name: count, dtype: int64

In [25]:
# Conversion de la colonne date en vrai format date
if "date" in df_observations_staging.columns:
    df_observations_staging["date_clean"] = pd.to_datetime(
        df_observations_staging["date"],
        errors="coerce"
    )

if "date" in df_pfas_long_staging.columns:
    df_pfas_long_staging["date_clean"] = pd.to_datetime(
        df_pfas_long_staging["date"],
        errors="coerce"
    )


# Nettoyage de l'année dans la table observation
year_original_obs = pd.to_numeric(
    df_observations_staging["year"],
    errors="coerce"
).astype("Int64")

year_from_date_obs = df_observations_staging["date_clean"].dt.year.astype("Int64")

df_observations_staging["year_clean"] = year_from_date_obs.fillna(year_original_obs)


# Nettoyage de l'année dans la table longue
year_original_long = pd.to_numeric(
    df_pfas_long_staging["year"],
    errors="coerce"
).astype("Int64")

year_from_date_long = df_pfas_long_staging["date_clean"].dt.year.astype("Int64")

df_pfas_long_staging["year_clean"] = year_from_date_long.fillna(year_original_long)


print("Années disponibles dans la table observation :")
display(df_observations_staging["year_clean"].describe())

Années disponibles dans la table observation :


count       929442.0
mean     2019.250647
std         4.019292
min           1977.0
25%           2016.0
50%           2020.0
75%           2023.0
max           2026.0
Name: year_clean, dtype: Float64

In [26]:
# Conversion des coordonnées en numérique
for table in [df_observations_staging, df_pfas_long_staging]:
    table["lat"] = pd.to_numeric(table["lat"], errors="coerce")
    table["lon"] = pd.to_numeric(table["lon"], errors="coerce")

    # Vérification de la validité des coordonnées
    table["has_valid_coordinates"] = (
        table["lat"].between(-90, 90) &
        table["lon"].between(-180, 180)
    )

print("Taux de coordonnées valides dans la table observation :")
print(round(df_observations_staging["has_valid_coordinates"].mean() * 100, 2), "%")

Taux de coordonnées valides dans la table observation :
95.59 %


In [34]:
# Conversion de pfas_value en numérique
df_pfas_long_staging["pfas_value"] = pd.to_numeric(
    df_pfas_long_staging["pfas_value"],
    errors="coerce"
)

# Conversion de pfas_sum en numérique
df_observations_staging["pfas_sum"] = pd.to_numeric(
    df_observations_staging["pfas_sum"],
    errors="coerce"
)

# Indicateurs simples
df_observations_staging["is_missing_pfas_sum"] = df_observations_staging["pfas_sum"].isna()
df_observations_staging["is_zero_pfas_sum"] = df_observations_staging["pfas_sum"] == 0
df_observations_staging["is_positive_pfas_sum"] = df_observations_staging["pfas_sum"] > 0

# Calcul du seuil extrême par matrice et unité
positive_obs = df_observations_staging[df_observations_staging["pfas_sum"] > 0].copy()

q99_by_group = (
    positive_obs
    .groupby(["matrix_standardized", "unit_standardized"])["pfas_sum"]
    .quantile(0.99)
    .reset_index()
    .rename(columns={"pfas_sum": "pfas_sum_q99_group"})
)

# Ajout du seuil q99 à la table observation
df_observations_staging = df_observations_staging.merge(
    q99_by_group,
    on=["matrix_standardized", "unit_standardized"],
    how="left"
)

# Une valeur est extrême si elle dépasse le q99 de son groupe
df_observations_staging["is_extreme_pfas_sum"] = (
    df_observations_staging["pfas_sum"] > df_observations_staging["pfas_sum_q99_group"]
)

display(df_observations_staging[
    ["pfas_sum", "matrix_standardized", "unit_standardized", "pfas_sum_q99_group", "is_extreme_pfas_sum"]
].head())

,pfas_sum,matrix_standardized,unit_standardized,pfas_sum_q99_group,is_extreme_pfas_sum
0,130.0,Surface water,ng/L,2439.92,False
1,200.0,Surface water,ng/L,2439.92,False
2,42400.0,Surface water,ng/L,2439.92,True
3,50.0,Surface water,ng/L,2439.92,False
4,580.0,Surface water,ng/L,2439.92,False


In [35]:
# Fonction pour transformer less_than en indicateur booléen
def convert_less_than_flag(x):
    if pd.isna(x):
        return False
    
    if isinstance(x, bool):
        return x
    
    s = str(x).strip().lower()
    
    if s in ["true", "1", "yes", "y", "oui", "<", "less than", "lt"]:
        return True
    
    return False


# Création d'un indicateur propre
if "pfas_less_than" in df_pfas_long_staging.columns:
    df_pfas_long_staging["pfas_less_than_flag"] = df_pfas_long_staging["pfas_less_than"].apply(
        convert_less_than_flag
    )

    display(df_pfas_long_staging["pfas_less_than_flag"].value_counts(dropna=False))

pfas_less_than_flag
False    21530916
Name: count, dtype: int64

In [29]:
# Vérification entre l'unité globale et l'unité détaillée
if "pfas_unit_standardized" in df_pfas_long_staging.columns:
    df_pfas_long_staging["unit_mismatch"] = (
        df_pfas_long_staging["unit_standardized"] != df_pfas_long_staging["pfas_unit_standardized"]
    )

    mismatch_rate = df_pfas_long_staging["unit_mismatch"].mean() * 100

    print(f"Taux d'incohérence entre unit et pfas_unit : {mismatch_rate:.2f}%")

    display(
        df_pfas_long_staging[
            df_pfas_long_staging["unit_mismatch"]
        ][["unit", "pfas_unit", "unit_standardized", "pfas_unit_standardized"]].head(20)
    )

Taux d'incohérence entre unit et pfas_unit : 0.00%


,unit,pfas_unit,unit_standardized,pfas_unit_standardized


In [31]:
# Résumé final des deux tables staging
quality_summary = pd.DataFrame({
    "table": ["pfas_observations_staging", "pfas_measurements_long_staging"],
    "rows": [df_observations_staging.shape[0], df_pfas_long_staging.shape[0]],
    "columns": [df_observations_staging.shape[1], df_pfas_long_staging.shape[1]],
    "valid_coordinates_rate_%": [
        round(df_observations_staging["has_valid_coordinates"].mean() * 100, 2),
        round(df_pfas_long_staging["has_valid_coordinates"].mean() * 100, 2)
    ]
})

display(quality_summary)

,table,rows,columns,valid_coordinates_rate_%
0,pfas_observations_staging,929442,22,95.59
1,pfas_measurements_long_staging,21530916,28,96.81


In [36]:
display(df_observations_staging.head(2))
display(df_pfas_long_staging.head(2))

,observation_id,dataset_id,dataset_name,name,country,city,lat,lon,year,date,...,is_positive_pfas_sum,is_zero_pfas_sum,has_valid_coordinates,unit_standardized,matrix_standardized,date_clean,year_clean,is_missing_pfas_sum,pfas_sum_q99_group,is_extreme_pfas_sum
0,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,...,True,False,True,ng/L,Surface water,NaT,2018,False,2439.92,False
1,2,10,15 textile facilities emitting PFAS,Tarkett,Belgium,Dendermonde,51.016507,4.088303,2017.0,NaN,...,True,False,True,ng/L,Surface water,NaT,2017,False,2439.92,False


,observation_id,dataset_id,dataset_name,name,country,city,lat,lon,year,date,...,pfas_isomer,pfas_less_than,unit_standardized,pfas_unit_standardized,matrix_standardized,date_clean,year_clean,has_valid_coordinates,pfas_less_than_flag,unit_mismatch
0,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,...,NaN,NaN,ng/L,ng/L,Surface water,NaT,2018,True,False,False
1,1,10,15 textile facilities emitting PFAS,Maes,Belgium,Zwevegem,50.808932,3.352552,2018.0,NaN,...,NaN,NaN,ng/L,ng/L,Surface water,NaT,2018,True,False,False


In [33]:
# Chemins de sortie
observations_output_path = STAGING_DIR / "pfas_observations_staging.parquet"
measurements_output_path = STAGING_DIR / "pfas_measurements_long_staging.parquet"

# Export de la table observations
df_observations_staging.to_parquet(
    observations_output_path,
    index=False,
    compression="snappy"
)

# Export de la table détaillée par substance
df_pfas_long_staging.to_parquet(
    measurements_output_path,
    index=False,
    compression="snappy"
)

print("Export terminé")
print("Table observations :", observations_output_path)
print("Table mesures longues :", measurements_output_path)

Export terminé
Table observations : ..\data\staging\pfas_observations_staging.parquet
Table mesures longues : ..\data\staging\pfas_measurements_long_staging.parquet


In [37]:
# Suppression des objets lourds inutiles
del df_long
del pfas_details

# Nettoyage mémoire
gc.collect()

print("Mémoire nettoyée")

Mémoire nettoyée
